# Sentiment Classifier — Naive Bayes (from scratch)

Interactive walkthrough of the project in Jupyter. **No ready-made libraries** — every import comes from our own `src/` modules; cleaning, tokenizing, stemming, vocabulary, vectorization and Naive Bayes are all hand-written.

Naive Bayes is trained on **raw word counts**, as the multinomial model assumes. The hand-written TF-IDF stage still exists (`SentimentPipeline(use_tfidf=True)`) but is off by default: idf gives *rare* words the largest weights, so a word seen once in training could dominate a prediction — confidently wrong instead of uncertain.

**Setup:** launch Jupyter from the **project root** (the folder containing `main.py` and `src/`) so that `from src...` and the `data/...` paths resolve. Then *Run All*.

## 1. Load data & train
Train on **imdb + amazon**, test on **yelp** as an unseen domain.

In [ ]:
from src.sentence_loader import SentenceDatasetLoader
from src.pipeline import SentimentPipeline
from src.metrics import accuracy, confusion_matrix
from src.naive_bayes import NEUTRAL_LABEL

LABEL_NAMES = {0: 'Negative', 1: 'Positive', NEUTRAL_LABEL: 'Neutral'}

imdb   = SentenceDatasetLoader('data/imdb_labelled.txt').load()
amazon = SentenceDatasetLoader('data/amazon_cells_labelled.txt').load()
yelp   = SentenceDatasetLoader('data/yelp_labelled.txt').load()

train_texts  = imdb[0] + amazon[0]
train_labels = imdb[1] + amazon[1]
test_texts, test_labels = yelp

# min_df=2 drops words seen in only one training sentence (noise, not signal);
# alpha is the Laplace smoothing strength (higher = rare words matter less).
pipe = SentimentPipeline(neutral_threshold=0.3, min_df=2, alpha=1.0)
pipe.train(train_texts, train_labels)

print(f'trained on {len(train_texts)} sentences | vocabulary: {pipe.vocab.size()} words')

## 2. Classify a sentence
`pipe.classify_text(text)` returns `(label, probs)`. The helper below prints it with a confidence bar.

In [4]:
def predict(text, width=24):
    label, probs = pipe.classify_text(text)
    print(f'\"{text}\"')
    print(f'  -> {LABEL_NAMES[label]}   (confidence {max(probs.values())*100:.0f}%)')
    for c in sorted(probs):
        filled = int(round(probs[c] * width))
        bar = '█' * filled + '░' * (width - filled)
        print(f'  {LABEL_NAMES[c]:<9} {bar} {probs[c]*100:5.1f}%')
    print()

predict('I absolutely loved this movie, it was fantastic!')
predict('The food was terrible and the service was awful.')
predict('it was okay i guess')

"I absolutely loved this movie, it was fantastic!"
  -> Positive   (confidence 100%)
  Negative  ░░░░░░░░░░░░░░░░░░░░░░░░   0.2%
  Positive  ████████████████████████  99.8%

"The food was terrible and the service was awful."
  -> Negative   (confidence 100%)
  Negative  ████████████████████████ 100.0%
  Positive  ░░░░░░░░░░░░░░░░░░░░░░░░   0.0%

"it was okay i guess"
  -> Negative   (confidence 70%)
  Negative  █████████████████░░░░░░░  69.8%
  Positive  ███████░░░░░░░░░░░░░░░░░  30.2%



### Try your own
Edit the string and re-run, or use the `input()` box.

In [ ]:
predict('the battery life is amazing but the screen is too dim')

In [ ]:
# Optional: type a sentence when prompted.
# Wrapped so 'Run All' / headless execution skips it instead of erroring.
try:
    predict(input('Enter a sentence: '))
except Exception:
    print('(no input available here — run this cell on its own to type a sentence)')

## 3. Evaluation on the Yelp test set
The test set has no neutral labels, so every Neutral prediction counts as wrong in overall accuracy — hence the separate confident-only number.

In [ ]:
predictions = pipe.predict(test_texts)

overall = accuracy(test_labels, predictions)
confident = [(t, p) for t, p in zip(test_labels, predictions) if p != NEUTRAL_LABEL]
neutral_count = len(predictions) - len(confident)

print(f'Overall accuracy          : {overall*100:5.1f}%')
if confident:
    c_acc = accuracy([t for t, _ in confident], [p for _, p in confident])
    print(f'Accuracy (confident only) : {c_acc*100:5.1f}%   (Neutral excluded)')
print(f'Neutral predictions       : {neutral_count} / {len(predictions)}')

## 4. Confusion matrix (pure-Python heatmap)

In [ ]:
matrix, labels = confusion_matrix(test_labels, predictions)
names = [LABEL_NAMES[l] for l in labels]

# Shade each cell by its share of the largest count — no plotting library.
def show_heatmap(matrix, names, width=7):
    biggest = max((v for row in matrix for v in row), default=1) or 1
    shades = ' ░▒▓█'                      # light -> dark
    label_w = max(len(n) for n in names)

    print('rows = true, cols = predicted   (denser block = larger count)\n')
    print(' ' * label_w + ''.join(f'{n:^{width+2}}' for n in names))
    for n, row in zip(names, matrix):
        cells = ''
        for v in row:
            level = round(v / biggest * (len(shades) - 1))
            cells += ' ' + shades[level] * width + ' '
        print(f'{n:>{label_w}}{cells}')

    print('\ncounts:')
    print(' ' * label_w + ''.join(f'{n:^{width+2}}' for n in names))
    for n, row in zip(names, matrix):
        print(f'{n:>{label_w}}' + ''.join(f'{v:^{width+2}}' for v in row))

show_heatmap(matrix, names)

## 5. Most informative words per class

In [ ]:
top_words = pipe.model.top_informative_words(pipe.vocab, top_n=12)
for c in pipe.model.classes:
    words = [w for w, _ in top_words[c]]
    print(f'{LABEL_NAMES[c]:>9}: ' + ', '.join(words))

## 6. Tune the Neutral threshold
Higher = more sentences flagged Neutral. Change it live and re-classify (no retraining needed).

In [ ]:
pipe.neutral_threshold = 0.5
predict('it was okay i guess')
pipe.neutral_threshold = 0.3  # restore